# Review information: title + abstract instead of research questions and criteria

Same benchmark, records, record input (title + abstract), question and model as [01_lgar_baseline.ipynb](01_lgar_baseline.ipynb). One factor changes: `state.review`.

| | `state.review` |
|---|---|
| baseline | title, research questions, inclusion criteria, exclusion criteria (LGAR) |
| this run | title, abstract of the Cochrane review (PubMed) |

The review abstract comes from the PMC link in the LGAR criteria file, mapped to its PMID.

Caveats:
- A Cochrane abstract has its own *Selection criteria* section. So this run removes the LGAR criteria and research questions, but the abstract still contains a short version of the criteria.
- *Main results* reports what the review found in its included studies (number of studies, participants, effects). This comes from the review's outcome, not from per-record labels, and no included study is named. Still, it is information a screener would not have before screening.
- The PMC version can be a later update of the review than the one the CLEF TAR labels were made for. The LGAR criteria have the same issue.
- The question text still mentions "inclusion criteria". It is kept the same so that only one factor changes.

Labels: CLEF TAR abstract-level (`label_abs`) and content-level (`label_content`).

Nothing paid runs unless `RUN_SMOKE`, `RUN_SMALL` or `RUN_PAID` is set to `True` below.

In [1]:
import json, sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "README.md").exists())
sys.path.insert(0, str(ROOT / "src"))
import screening as S

RUN_SMOKE = True   # one paid call
RUN_SMALL = True    # one small review (SMALL_REVIEW), about $0.003
RUN_PAID = True    # all 20 reviews; read the cost estimate first
SMALL_REVIEW = "CD012164"   # 61 records, 7 abstract-level and 3 content-level includes
OUT = S.out_dir("review_title_abstract")
BASELINE = S.out_dir("baseline")

/Users/joshua/Desktop/Repositories/Research/decision-model-screener/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data

In [2]:
records, CRITERIA = S.load_records()
records = S.add_bm25(records, CRITERIA)
REVIEW_ABS = S.fetch_review_abstracts(CRITERIA)

check = pd.DataFrame({
    "lgar_title": {r: c["title"] for r, c in CRITERIA.items()},
    "pubmed_title": {r: a["title"] for r, a in REVIEW_ABS.items()},
    "abstract_chars": {r: len(a["abstract"]) for r, a in REVIEW_ABS.items()},
    "sections": {r: ", ".join(l.split(":")[0] for l in a["abstract"].split("\n")) for r, a in REVIEW_ABS.items()},
})
assert (check.abstract_chars > 0).all(), "a review has no abstract"
check

20 reviews, 41996 records, 41183 unique PMIDs
not returned by PubMed: 0 | no abstract: 2183 of 41996


,lgar_title,pubmed_title,abstract_chars,sections
CD000996,Inhaled corticosteroids for bronchiectasis,Inhaled corticosteroids for bronchiectasis.,3895,"BACKGROUND, OBJECTIVES, SEARCH METHODS, SELECT..."
CD001261,Vaccines for preventing typhoid fever,Vaccines for preventing typhoid fever.,5834,"BACKGROUND, OBJECTIVES, SEARCH METHODS, SELECT..."
CD004414,Interventions for preventing occupational irri...,Interventions for preventing occupational irri...,6453,"BACKGROUND, OBJECTIVES, SEARCH METHODS, SELECT..."
CD006468,Anticoagulation for people with cancer and cen...,Anticoagulation for people with cancer and cen...,5497,"BACKGROUND, OBJECTIVES, SEARCH METHODS, SELECT..."
CD007867,Prescribed hypocaloric nutrition support for c...,Prescribed hypocaloric nutrition support for c...,5124,"BACKGROUND, OBJECTIVES, SELECTION CRITERIA, DA..."
CD009069,Prophylactic vaccination against human papillo...,Prophylactic vaccination against human papillo...,6870,"BACKGROUND, OBJECTIVES, SEARCH METHODS, SELECT..."
CD009642,Continuous intravenous perioperative lidocaine...,Continuous intravenous perioperative lidocaine...,6523,"BACKGROUND, OBJECTIVES, SEARCH METHODS, SELECT..."
CD010038,Face-to-face interventions for informing or ed...,Face-to-face interventions for informing or ed...,6273,"BACKGROUND, OBJECTIVES, SEARCH METHODS, SELECT..."
CD010239,Lower versus higher oxygen concentrations titr...,Lower versus higher oxygen concentrations titr...,4232,"BACKGROUND, OBJECTIVES, SEARCH METHODS, SELECT..."
CD010558,Psychological therapies for treatment-resistan...,Psychological therapies for treatment-resistan...,6558,"BACKGROUND, OBJECTIVES, SEARCH METHODS, SELECT..."


## Decision model input

In [3]:
def build_body(review, title, abstract):
    return {
        "model": S.MODEL,
        "state": {
            "review": {
                "title": CRITERIA[review]["title"],
                "abstract": REVIEW_ABS[review]["abstract"],
            },
            "record": {"title": title, "abstract": abstract},
        },
        "questions": S.QUESTIONS,
    }

print(json.dumps(build_body(*records.loc[0, ["review", "title", "abstract"]]), indent=1)[:3000])

{
 "model": "typesafe/jev-1.14",
 "state": {
  "review": {
   "title": "Inhaled corticosteroids for bronchiectasis",
   "abstract": "BACKGROUND: Bronchiectasis is being increasingly diagnosed and recognised as an important contributor to chronic lung disease in both adults and children in high- and low-income countries. It is characterised by irreversible dilatation of airways and is generally associated with airway inflammation and chronic bacterial infection. Medical management largely aims to reduce morbidity by controlling the symptoms, reduce exacerbation frequency, improve quality of life and prevent the progression of bronchiectasis. This is an update of a review first published in 2000.\nOBJECTIVES: To evaluate the efficacy and safety of inhaled corticosteroids (ICS) in children and adults with stable state bronchiectasis, specifically to assess whether the use of ICS: (1) reduces the severity and frequency of acute respiratory exacerbations; or (2) affects long-term pulmonary 

## Cost estimate (no API call)

In [4]:
S.estimate_cost(records, build_body)

TypeSafe | typesafe/jev-1.14-20261014 | USD per 1M input tokens: 0.041999999999999996 | output: 0
requests: 41996 | mean chars per request: 7604 | max: 17284
estimated input tokens: 104.4M | estimated cost: $4.38


,records,chars,tokens_M,usd
review,,,,
CD000996,281,1577675,0.516,0.022
CD001261,571,4319429,1.412,0.059
CD004414,336,2785153,0.910,0.038
CD006468,3874,28323503,9.256,0.389
CD007867,943,6868919,2.245,0.094
CD009069,1757,15654675,5.116,0.215
CD009642,1922,16524903,5.400,0.227
CD010038,8867,74958927,24.496,1.029
CD010239,224,1462607,0.478,0.020


## Smoke test (one paid call)

In [5]:
if RUN_SMOKE:
    resp = S.call(build_body(*records.loc[0, ["review", "title", "abstract"]]))
    print(json.dumps(resp, indent=1))

{
 "model": "typesafe/jev-1.14-20261014",
 "answers": {
  "include": {
   "type": "noul",
   "noul": 0.03
  }
 },
 "usage": {
  "input_tokens": 1555,
  "output_tokens": 20,
  "cost": 6.531e-05,
  "is_byok": false,
  "cost_details": {
   "upstream_inference_cost": 6.531e-05,
   "upstream_inference_prompt_cost": 6.531e-05,
   "upstream_inference_completions_cost": 0
  }
 },
 "id": "gen-dec-1794909978-Z3OPESUVS38jygWeUdLU",
 "provider": "TypeSafe"
}


## One small review (paid)

In [6]:
if RUN_SMALL:
    S.run_reviews(records, [SMALL_REVIEW], build_body, OUT)


reviews:   0%|          | 0/1 [00:00<?, ?it/s]


CD012164:   0%|          | 0/61 [00:00<?, ?it/s]


CD012164:   2%|▏         | 1/61 [00:00<00:15,  4.00it/s]


CD012164:   7%|▋         | 4/61 [00:00<00:05, 11.24it/s]


CD012164:  10%|▉         | 6/61 [00:00<00:04, 11.12it/s]


CD012164:  15%|█▍        | 9/61 [00:00<00:04, 11.95it/s]

... (12 outputs left out)


CD012164:  95%|█████████▌| 58/61 [00:02<00:00, 24.28it/s]


CD012164: 100%|██████████| 61/61 [00:03<00:00, 20.21it/s]


reviews: 100%|██████████| 1/1 [00:03<00:00,  3.23s/it]


reviews: 100%|██████████| 1/1 [00:03<00:00,  3.23s/it]

## Full run (paid)

In [6]:
if RUN_PAID:
    S.run_reviews(records, CRITERIA, build_body, OUT)

reviews: 100%|██████████| 20/20 [21:37<00:00, 64.86s/it] 


## Results

Both configurations are scored on the same records: the reviews that have a cache file for this run. Failed calls and refusals get the lowest score and are counted. Baseline scores come from the baseline cache, so nothing is re-requested.

In [7]:
done = sorted(p.stem for p in OUT.glob("*.jsonl"))
print(len(done), "of", len(CRITERIA), "reviews run:", done)
sub = records[records.review.isin(done)].copy()
sub = S.add_scores(sub, BASELINE, "baseline")
sub = S.add_scores(sub, OUT, "title_abstract")

results = {c: S.evaluate(sub, c) for c in ("bm25", "baseline", "title_abstract")}
results["difference"] = results["title_abstract"] - results["baseline"]
results["difference"]["reviews"] = results["baseline"]["reviews"]
pd.concat(results)

20 of 20 reviews run: ['CD000996', 'CD001261', 'CD004414', 'CD006468', 'CD007867', 'CD009069', 'CD009642', 'CD010038', 'CD010239', 'CD010558', 'CD010753', 'CD011140', 'CD011571', 'CD011768', 'CD011977', 'CD012069', 'CD012164', 'CD012342', 'CD012455', 'CD012551']
baseline: scored 41996 of 41996 records | model: {'typesafe/jev-1.14-20261014'} | cost: $1.80 | without a score (ranked last): 0
title_abstract: scored 41996 of 41996 records | model: {'typesafe/jev-1.14-20261014'} | cost: $3.61 | without a score (ranked last): 0


MAP  R@1%  R@5%  R@10%  R@20%  R@50%  WSS@95%  \
bm25           label_abs      22.3  12.2  34.4   46.2   60.7   82.6     33.3   
               label_content  17.0  12.0  30.2   47.0   65.2   86.7     43.6   
baseline       label_abs      57.9  31.5  65.0   77.8   86.8   97.0     66.3   
               label_content  60.7  45.3  84.2   90.9   94.9   98.5     81.0   
title_abstract label_abs      61.5  32.7  65.4   78.9   88.8   98.4     70.1   
               label_content  71.3  48.9  84.9   92.8   96.2   99.5     83.5   
difference     label_abs       3.6   1.2   0.4    1.1    2.0    1.4      3.8   
               label_content  10.6   3.6   0.7    1.9    1.3    1.0      2.5   

                              TNR@95%  WSS@100%  reviews  
bm25           label_abs         39.5      28.7     20.0  
               label_content     49.5      44.6     18.0  
baseline       label_abs         74.7      62.1     20.0  
               label_content     88.3      81.4     18.0  
title_abstract label_abs         78.8      66.3     20.0  
               label_content     91.1      85.2     18.0  
difference     label_abs          4.1       4.2     20.0  
               label_content      2.8       3.8     18.0

In [8]:
THRESHOLD = 0.05   # same as baseline
summaries, per_review = {}, {}
for c in ("baseline", "title_abstract"):
    summaries[c], per_review[c] = S.threshold_table(sub, c, THRESHOLD)
print(f"threshold = {THRESHOLD}")
display(pd.concat(summaries))
display(pd.concat(per_review, axis=1))

threshold = 0.05


label_abs                      \
                                  sensitivity specificity flagged   
baseline       mean over reviews         92.3        75.6    27.8   
               all records pooled        89.9        77.7    23.8   
title_abstract mean over reviews         97.0        63.3    39.6   
               all records pooled        98.8        69.0    32.5   

                                  label_content                      
                                    sensitivity specificity flagged  
baseline       mean over reviews           98.0        74.0    27.8  
               all records pooled          91.8        77.1    23.8  
title_abstract mean over reviews          100.0        61.9    39.6  
               all records pooled          99.8        68.3    32.5

baseline                                                        \
           label_abs                     label_content                       
         sensitivity specificity flagged   sensitivity specificity flagged   
CD000996        88.9        82.4    19.9         100.0        81.8    19.9   
CD001261        91.7        65.7    41.5          90.9        60.5    41.5   
CD004414       100.0        88.1    16.1         100.0        85.7    16.1   
CD006468        94.2        73.2    27.7         100.0        72.6    27.7   
CD007867       100.0        51.6    49.3         100.0        51.2    49.3   
CD009069        76.9        82.0    20.7         100.0        79.6    20.7   
CD009642        98.4        83.7    19.0          98.0        83.1    19.0   
CD010038       100.0        90.8     9.5         100.0        90.6     9.5   
CD010239        91.7        95.3     9.4         100.0        94.9     9.4   
CD010558        97.3        78.9    22.1         100.0        78.2    22.1   
CD010753       100.0        42.8    57.7         100.0        42.6    57.7   
CD011140       100.0        97.9     3.5           NaN        96.5     3.5   
CD011571        93.3        89.3    19.2         100.0        84.3    19.2   
CD011768       100.0        83.5    17.0         100.0        83.3    17.0   
CD011977        89.8        76.0    40.5          89.2        70.9    40.5   
CD012069        85.3        38.6    63.6          85.7        38.1    63.6   
CD012164       100.0        42.6    62.3         100.0        39.7    62.3   
CD012342        50.0        92.3     7.8           NaN        92.2     7.8   
CD012455       100.0        97.7     2.8         100.0        97.5     2.8   
CD012551        88.2        59.1    46.4         100.0        56.4    46.4   

         title_abstract                                                        
              label_abs                     label_content                      
            sensitivity specificity flagged   sensitivity specificity flagged  
CD000996           88.9        71.7    30.2         100.0        71.3    30.2  
CD001261           98.6        48.3    57.6         100.0        44.1    57.6  
CD004414          100.0        80.9    22.9         100.0        78.7    22.9  
CD006468           98.1        65.1    35.8         100.0        64.4    35.8  
CD007867          100.0        33.7    66.9         100.0        33.4    66.9  
CD009069           97.4        66.2    36.6         100.0        63.6    36.6  
CD009642          100.0        70.6    31.6         100.0        70.2    31.6  
CD010038          100.0        89.7    10.6         100.0        89.5    10.6  
CD010239          100.0        66.0    37.5         100.0        65.4    37.5  
CD010558          100.0        44.8    55.8         100.0        44.4    55.8  
CD010753          100.0        51.5    49.1         100.0        51.3    49.1  
CD011140          100.0        94.7     6.6           NaN        93.4     6.6  
CD011571           93.3        87.8    20.5         100.0        82.9    20.5  
CD011768          100.0        84.1    16.4         100.0        83.8    16.4  
CD011977          100.0        62.3    53.3         100.0        57.6    53.3  
CD012069           99.4        14.8    86.5          99.6        14.5    86.5  
CD012164          100.0        38.9    65.6         100.0        36.2    65.6  
CD012342           66.7        64.5    35.6           NaN        64.4    35.6  
CD012455          100.0        92.3     8.1         100.0        92.2     8.1  
CD012551           98.5        38.6    65.7         100.0        36.1    65.7